## Evaluation of Generated Clinical Notes using BERTScore and BERT Embeddings

This notebook evaluates the generated clinical notes by comparing them to the reference notes using the BERTScore evaluation metric. The BERTScore uses cosine similarity between BERT embeddings to calculate precision, recall, and F1 scores for each sentence. 

The results from this evaluation will provide insights into the quality of the generated clinical notes based on semantic similarity to the reference notes.

<hr style="height:1rem;">

## Imports

In [1]:
import os
import re

from pathlib import Path
import pandas as pd

from bert_score import score
from transformers import BertModel, BertTokenizer
from transformers import BertForPreTraining

# pytorch library
import torch # the main pytorch library
import torch.nn.functional as f # the sub-library containing different functions for manipulating with tensors

import nltk
from nltk.tokenize import sent_tokenize

In [2]:
import warnings
warnings.filterwarnings("ignore")

import logging
from transformers import logging as transformers_logging

# Suppress specific logs from transformers
transformers_logging.set_verbosity_error()

In [3]:
# Download necessary NLTK data
nltk.download('punkt')

[nltk_data] Downloading package punkt to D:\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

<hr style="height:1rem;">

## Path Settings

In [4]:
NOTEBOOK_PATH = os.getcwd()
NOTEBOOK_DATA_PATH = os.path.join(
    NOTEBOOK_PATH,
    'data'
)
NOTEBOOK_EVALUATION_PATH = os.path.join(
    NOTEBOOK_PATH,
    'evaluation'
)
NOTEBOOK_RESULTS_PATH = os.path.join(
    NOTEBOOK_PATH,
    'results'
)
NOTEBOOK_PLOTS_PATH = os.path.join(
    NOTEBOOK_PATH,
    'plots'
)
DISSERTATION_PATH = Path.cwd().resolve().parents[0]
DISSERTATION_DATA_PATH = os.path.join(
    DISSERTATION_PATH,
    'data'
)

print(NOTEBOOK_PATH)
print(NOTEBOOK_DATA_PATH)
print(NOTEBOOK_RESULTS_PATH)
print(NOTEBOOK_EVALUATION_PATH)
print(NOTEBOOK_PLOTS_PATH)
print(DISSERTATION_PATH)
print(DISSERTATION_DATA_PATH)

D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples
D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples\data
D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples\results
D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples\evaluation
D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples\plots
D:\ODU\03 - Dissertation Defense\COT-KG\latest
D:\ODU\03 - Dissertation Defense\COT-KG\latest\data


<hr style="height:1rem;">

## Device Setting

With pytorch we are able to move the python calculations to the GPU. To do this we define the device on which we wish to run the calculations. Depending if cuda (the GPU drivers that enable running calculations on the graphic card) is enabled on the machine, we define the device as follows:

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


<hr style="height:1rem;">

## Variable Declarations

In [6]:
ROBERTA_BASE = "roberta-base"
MICROSOFT_DEBERTA_XL_MNLI = "microsoft/deberta-xlarge-mnli"
BERT_UNCASED = "bert-base-uncased"
BERT_LARGE_UNCASED = "bert-large-uncased"

# BioBERT, SciBERT, and ClinicalBERT
BIOBERT_BASED_CASED_V1_1 = "dmis-lab/biobert-base-cased-v1.1"
BIOBERT_BASED_CASED_V1_2 = "dmis-lab/biobert-base-cased-v1.2"
SCIBERT_SCIVOCAB_UNCASED = "allenai/scibert_scivocab_uncased"
CLINICAL_BERT = "emilyalsentzer/Bio_ClinicalBERT"

MODEL_TYPE = BERT_LARGE_UNCASED
print(MODEL_TYPE)

bert-large-uncased


<hr style="height:1rem;">

## Function Declarations

In [7]:
def parse_sentences(text):
    """
    Tokenize the input text into individual sentences.

    This function uses NLTK's sentence tokenizer to segment the input text into sentences.

    Parameters:
    - text (str): The input text to be tokenized into sentences.

    Returns:
    - List[str]: A list of sentences from the input text.
    """
    # Use NLTK's sentence tokenizer to segment the input text into sentences
    return sent_tokenize(text)

In [8]:
def align_sentences(ground_truth_sentences, generated_sentences):
    """
    Align sentences from ground truth and generated outputs using a simple positional strategy.

    This method truncates both lists to the length of the shorter one, 
    assuming that sentence order and count roughly match.

    Parameters:
    - ground_truth_sentences (List[str]): List of reference sentences.
    - generated_sentences (List[str]): List of machine-generated sentences.

    Returns:
    - Tuple (List[str], List[str]): Aligned ground truth and generated sentences of equal length.
    """
    
    # Determine the length of the shorter list to avoid index errors
    min_len = min(len(ground_truth_sentences), len(generated_sentences))

    # Truncate both lists to the same length
    aligned_ground_truth = ground_truth_sentences[:min_len]
    aligned_generated = generated_sentences[:min_len]
    
    # Return the aligned sentence lists
    return aligned_ground_truth, aligned_generated    

In [9]:
def get_bertscore(ground_truth, machine_generated):
    """
    Compute the average BERTScore (Precision, Recall, F1) between ground truth and machine-generated text.

    Parameters:
    - ground_truth (str): The reference text written by a human.
    - machine_generated (str): The text generated by a machine (e.g., an LLM).

    Returns:
    - Tuple (float, float, float): Mean Precision, Recall, and F1 BERTScore across aligned sentence pairs.
    """
    
    # Split the ground truth text into individual sentences
    ground_truth_sentences = parse_sentences(ground_truth)

    # Split the machine-generated text into individual sentences
    generated_sentences = parse_sentences(machine_generated)
    
    # Align the ground truth and generated sentences for one-to-one comparison
    aligned_ground_truth, aligned_generated = align_sentences(ground_truth_sentences, generated_sentences)
    
    # Compute BERTScore precision (P), recall (R), and F1 score (F1)
    P, R, F1 = score(
        aligned_generated,                           # Hypothesis sentences (model outputs)
        aligned_ground_truth,                        # Reference sentences (ground truth)
        lang = "en",                                 # Language setting for BERTScore
        #batch_size = 64,                             # Optional: control memory and speed
        #nthreads = 4,                                # Optional: use multiple CPU threads
        #num_layers = 10,                             # Optional: choose BERT layer (default is usually the last)      
        model_type = MODEL_TYPE,                     # Predefined model type for embeddings (e.g., "bert-large-cased")
        verbose = False                              # Suppress verbose logging
    )
    
    return P.mean().item(), R.mean().item(), F1.mean().item()

In [10]:
def get_document_bertscore(ground_truth, machine_generated):
    """
    Compute the BERTScore (Precision, Recall, F1) between full ground truth and machine-generated documents.

    This function treats the entire input texts as single documents rather than sentence-aligned segments.

    Parameters:
    - ground_truth (str): The reference human-written text.
    - machine_generated (str): The machine-generated text to evaluate.

    Returns:
    - Tuple (float, float, float): Mean BERTScore Precision, Recall, and F1 across all text segments.
    """

    # Calculate BERTScore directly on the full document texts (as single-element lists)
    P, R, F1 = score(
        [machine_generated],       # the machine output as a single-element list
        [ground_truth],            # the ground truth similarly
        lang = "en",              # Language setting for the BERT model
        # batch_size = 64,        # Optional: size of mini-batches
        # nthreads = 4,           # Optional: number of threads for processing
        # num_layers = 10,        # Optional: specific BERT layer to use for embeddings
        model_type = MODEL_TYPE, # Pre-trained BERT model type (e.g., "bert-large-cased")
        verbose = False          # Disable verbose output
    )
    
    # Return the mean precision, recall, and F1 score as scalar floats
    return P.mean().item(), R.mean().item(), F1.mean().item()

In [11]:
def compute_cosine_distances(ground_truth_text, generated_text, model, tokenizer, device):
    # Due to the token size of our GPU - 512 tokens, we need to trim off (256 for ground-truth text and 256 for generated text)    
    tokens = tokenizer.tokenize(generated_text)
    if len(tokens) > 256:
        # Truncate to the maximum token length
        tokens = tokens[:256]

        # Convert the tokens back to a string
        generated_text = tokenizer.convert_tokens_to_string(tokens)         
    
    texts  = [
        ground_truth_text,
        generated_text
    ]

    # Get the encodings of the sentences. Since the sentences are of different lenghts we set the 'padding' parameter 
    # to 'True' which will pad the shorter sentences. We also set the return_tensors to 'pt' so that we get the 'pytorch' 
    # tensors instead of python lists.
    encodings = tokenizer(
        texts, # the texts to be tokenized
        padding=True, # pad the texts to the maximum length (so that all outputs have the same length)
        return_tensors='pt' # return the tensors (not lists)
    )

    # The input tensors need to be on the same device as the model. In order for us to use the BERT model to get the 
    # sentence embeddings we must first move the input tensors to the same device as the model. 
    encodings = encodings.to(device)

    # In order to get the token embeddings we will send the encodings through the model. Pytorch by default calculates 
    # the gradients for every layer of the model which is really handy when training the model but not when we want to 
    # use it in production. Gradients take additional space on CPU or GPU which you don't want to populate with things 
    # you don't need.  We can disable gradient calculation by adding the 'with torch.no_grad():' before the model.
    
    # disable gradient calculations
    with torch.no_grad():
        # get the model embeddings
        embeds = model(**encodings)    
    
    # The output of the model is a tuple of tensors:
    #   The first tensor (embeds[0]) contains the embeddings of the sentence tokens.
    #   The second tensor (embeds[1]) contains the embeddings of the '[CLS]' token that can be then sent further to 
    #     train the next sentence prediction. We won't use this tensor in this notebook.
    embeds = embeds[0]
    
    # Way 1: Use the [CLS] Embeddings
    # We will take the first embedding representing the '[CLS]' token of all three sentences and calculate the 
    # cosine distance.
    CLSs = embeds[:, 0, :]
    
    # normalize the CLS token embeddings
    normalized = f.normalize(CLSs, p=2, dim=1)
    
    # calculate the cosine similarity
    cls_dist = normalized.matmul(normalized.T).cpu()
    cls_dist = cls_dist.new_ones(cls_dist.shape) - cls_dist
    cls_dist = cls_dist.numpy()    
    
    # Way 2: Computing the Mean of All Output Vectors
    # We will calculate the mean of all embeddings for each sentence and calculate the cosine distance. 
    # To get the means we will use the '.mean' method that is documented here:
    #     https://pytorch.org/docs/stable/generated/torch.mean.html#torch.mean
    MEANS = embeds.mean(dim=1)
    
    # normalize the MEANS token embeddings
    normalized = f.normalize(MEANS, p=2, dim=1)
    
    # calculate the cosine similarity
    mean_dist = normalized.matmul(normalized.T).cpu()
    mean_dist = mean_dist.new_ones(mean_dist.shape) - mean_dist
    mean_dist = mean_dist.numpy()
    
    # Way 3: Compute the max-over-time of the Output Vectors
    # We will calculate the max-over-time of all embeddings for each sentence and calculate the cosine distance. 
    # The max-over-time is basically getting the maximum value for each dimension of the tokens. 
    # We will use the .max method that is documented here: 
    #     https://pytorch.org/docs/stable/generated/torch.max.html#torch.max
    # The .max method returns a tuple where the first value is a tensor of maximum values based on the dim argument, 
    # and the second value are the indices of the index location of each maximum value found ('argmax').
    MAXS, _ = embeds.max(dim=1)
    
    # normalize the MEANS token embeddings
    normalized = f.normalize(MAXS, p=2, dim=1)
    
    # calculate the cosine similarity
    max_dist = normalized.matmul(normalized.T).cpu()
    max_dist = max_dist.new_ones(max_dist.shape) - max_dist
    max_dist = max_dist.numpy()

    # Return the lists of distances
    return cls_dist[0][1], mean_dist[0][1], max_dist[0][1]

<hr style="height:1rem;">

## Read Data

### Read Ground-Truth Data from CodiEsp

In [12]:
codiesp_file = "../data/samples/codiesp_snomedct_samples.csv"
codiesp_df = pd.read_csv(codiesp_file)
codiesp_df['icd_codes'] = codiesp_df['icd_codes'].apply(lambda x: ' '.join([code.split('.')[0] for code in x.split()]))
len(codiesp_df)
codiesp_df.head(5)

,article_id,text_references,icd_codes,age,gender,text,icd_match_count,total_icd_count,bin
0,S2340-98942015000100005-1,metastatic liver lesion; vomiting; disease; re...,C78 R11 R69 0DTP 0FB0 R06 C56 R97 R55,62,female,We report the case of a 62-year-old patient wh...,7,9,bin7
1,S1889-836X2016000100006-1,osteoporosis; otosclerosis; elbow fractures; s...,M81 H80 S42 S42 Z87 R69 S43 BW40ZZZ N20 BR00ZZ...,65,male,A 65-year-old male patient diagnosed with OI f...,8,11,bin8
2,S1888-75462015000400006-1,edema; pain; bilateral hip MRI; plain radiolog...,R60 R52 BQ31 BQ01 M89 BQ30 I96 M25 0SJ94ZZ M25...,38,male,A 38-year-old male patient was evaluated in co...,9,14,bin9
3,S1699-695X2017000100011-1,threatened labor; Mental retardation; delivery...,O47 F79 10D07Z3 G40 10E0XZZ F84 F71 G47 F40 R1...,12,male,The patient is a 12-year-old male resulting fr...,10,12,bin10
4,S1139-76322009000100006-1,tachycardia; supraventricular tachycardia; ECG...,R00 I47 4A02X4Z R23 R21 R06 10E0XZZ R23,29-day,female,"A 29-day-old newborn weighing 4,100 kg was tak...",6,8,bin6


<hr style="height:1rem;">

## Metrics Evaluations

### METRIC #1 - Sentence Similarity using BERT

Calculate the similarity of different documents using BERT. Two documents will be sent through the BERT model to retrieve its embeddings and use them to calculate the document similarity by:

- Using the [CLS] token of each sentence.
- Calculating the mean of sentence embeddings.
- Calculating the max-over-time of the sentence embeddings.

The python libraries that we will use in this notebook are:

- transformers. The huggingface library containing the general-purpose architectures for NLP.
- torch. An open source machine learning framework used to create and train models on the GPU.
- matplotlib. A comprehensive library for creating static, animated and interactive visualizations in Python.

#### Measure the Sentence Similarities

Using the sentence token embeddings, we will calculate the sentence similarities. We will do this in three ways:

- Using the ```[CLS]``` token embeddings.
- Calculating the mean of all sentence token embeddings.
- Calculating the max-over-time of the sentence token embeddings.

In all three ways we will calculate the sentence similarities using the cosine distance.

<hr style="height:0.25rem;">

#### BERT Tokenizer and Model

Now we will import a pretrained BERT model from Hugging Face. There are a lot of different BERT models that were trained with different hyperparameters and tasks. A full list of official models is available [here](https://huggingface.co/transformers/pretrained_models.html), but there are also a lot of models that are uploaded by the [community](https://huggingface.co/models).

##### BERT Model

In this notebook we will use the ```bert-large-cased``` model. To use it we will need to import both the ```tokenizer``` and the ```model```: The ```tokenizer``` will enable us to transform strings into tensors that can be then sent to the model, which in turn will give us the embeddings.

##### BERT Tokenizer

The tokenizer transforms a string into lists, tensors, etc. which can be used as the input of the model. Different tokenizers have different inputs. The BERT tokenizer takes the following inputs:

- ```text```. The sequence or batch of sequences to be encoded.
- ```text_pair```. The OTHER sequence of batch of sequences to be encoded (used for next sentence prediction). (Default: None)
- ```padding```. Activates and controls padding. If True, pad to the longest sequence in the batch (or no padding if only a single sequence is provided). Other options are also available. (Default: False)
- ```truncation```. Activates and controls truncation. If True, Truncate to a maximum length specified with the ```max_length``` argument. (Default: False)
- ```max_length```. Controls the maximum length to use by one of the truncation/padding parameters.
return_tensors. If set, will return tensors instead of list of python integers.

The full documentation of the BERT tokenizer can be found [here](https://huggingface.co/transformers/model_doc/bert.html#berttokenizer). NOTE: The BERT tokenizer inherits from the ```PreTrainedTokenizer``` class, which is documented [here](https://huggingface.co/transformers/main_classes/tokenizer.html#transformers.PreTrainedTokenizer).


The BERT tokenizer returns three values:

- ```input_ids```. The list/tensor of token ids for each sentence. The id ```101``` represents the class token ```[CLS]```, ```102``` represents the separator token ```[SEP]```, and 0 represents the padding token ```[PAD]```.
- ```attention_mask```. This list/tensor represents which ids to use when generating the tokens (e.g. ignores the [PAD] tokens).
- ```token_type_ids```. This list/tensor represents which tokens correspond to the first and second sentence (used for next sentence prediction).

<hr style="height:0.5rem;">

### METRIC #2 - BERTScore

####  Evaluating using BERTScore
ROUGE relies on the exact presence of words in both the predicted and reference texts, failing to interpret the underlying semantics. This is where [BERTScore](https://arxiv.org/abs/1904.09675) comes in and leverages the contextual embeddings from the BERT model, aiming to evaluate the similarity between a predicted and a reference sentence in the context of machine-generated text. By comparing embeddings from both sentences, `BERTScore` captures semantic similarities that might be missed by traditional n-gram based metrics.

#### Supported Models:
https://docs.google.com/spreadsheets/d/1RKOVpselB98Nnh_EOC4A2BYn8_201tmPODpNWu4w7xI/edit?gid=0#gid=0

In [13]:
# Load the pre-trained BERT model (without any task-specific head)
model = BertModel.from_pretrained(BERT_LARGE_UNCASED)
#model = BertForPreTraining.from_pretrained(BERT_LARGE_UNCASED)

# Load the tokenizer for the specified BERT model
tokenizer = BertTokenizer.from_pretrained(BERT_LARGE_UNCASED)

# Set the model to evaluation mode (disables dropout and gradient updates)
model = model.eval()

# Move the model to the specified device (e.g., CPU or GPU)
model = model.to(device)

In [14]:
ARTICLE_IDS = [
    'S2340-98942015000100005-1', # 0
    'S1889-836X2016000100006-1', # 1
    'S1888-75462015000400006-1', # 2
    'S1699-695X2017000100011-1', # 3
    'S1139-76322009000100006-1', # 4
    'S1139-13752009000200010-1', # 5
    'S1130-63432014000300013-4', # 6
    'S0376-78922009000400010-1', # 7
    'S0365-66912007000200011-1', # 8
    'S0004-06142007000600016-2', # 9
    'S0004-06142006000700014-1', # 10
    'S0004-06142006000600012-1', # 11
    'S0004-06142005001000016-1', # 12
    'S1887-85712015000200005-1', # 13
    'S0004-06142008000700011-1', # 14
    'S1888-75462017000100042-1', # 15
    'S1130-05582016000400228-1', # 16
    'S1130-05582009000400005-1', # 17
    'S1130-01082008001100009-1', # 18
    'S1130-01082007001000017-1', # 19
    'S1130-01082006001000017-1', # 20
    'S0365-66912011000400005-2', # 21
    'S0365-66912007000900016-1', # 22
    'S0365-66912004001200010-2', # 23
    'S0213-12852007000100002-1', # 24
    'S0210-56912010000200009-1', # 25
    'S0210-56912008000200007-1'  # 26
]

In [15]:
match_article_id = ARTICLE_IDS[1]
print(match_article_id)

generated_text = """
A 38-year-old male with established osteoradionecrosis of the facial bones presents with a three-week history of progressive left mandibular pain and swelling, noted to have occasional exposed bone and tenderness to palpation. Over the past two months he has experienced right wrist discomfort characterized by a sensation of instability and palpable crepitus with flexion and gripping activities. He reports a chronic history of bilateral pes planus with peroneal muscle spasm, but two weeks ago developed escalating pain, swelling, and purplish discoloration of his right forefoot despite offloading. That lesion evolved into a dry necrotic area over the second and third toes, for which he underwent percutaneous endoscopic drainage of a presumed abscess two days ago. He also describes intermittent swelling of the fingers and diffuse body aches over the last several weeks. He denies fevers, chills, weight loss, or history of recent trauma.
"""
print(generated_text)

S1889-836X2016000100006-1

A 38-year-old male with established osteoradionecrosis of the facial bones presents with a three-week history of progressive left mandibular pain and swelling, noted to have occasional exposed bone and tenderness to palpation. Over the past two months he has experienced right wrist discomfort characterized by a sensation of instability and palpable crepitus with flexion and gripping activities. He reports a chronic history of bilateral pes planus with peroneal muscle spasm, but two weeks ago developed escalating pain, swelling, and purplish discoloration of his right forefoot despite offloading. That lesion evolved into a dry necrotic area over the second and third toes, for which he underwent percutaneous endoscopic drainage of a presumed abscess two days ago. He also describes intermittent swelling of the fingers and diffuse body aches over the last several weeks. He denies fevers, chills, weight loss, or history of recent trauma.



<hr style="height:0.5rem;">

### Evaluate the Results


In [16]:
# Loop through a predefined number of runs
for index, row in codiesp_df.iterrows():
    article_id=str(row['article_id']) # used for title and filename

    if match_article_id != article_id:
        #print(f"Skipping (in IGNORE list): {article_id}")
        continue

    orig_generated_text = generated_text
    orig_ground_truth_text=str(row['text']) # used for title and filename
    print('\narticle_id:', article_id)
        
    # Due to the token size of our GPU - 512 tokens, we need to trim off (256 for ground-truth text and 256 for generated text)    
    ground_truth_tokens = tokenizer.tokenize(orig_ground_truth_text)
    if len(ground_truth_tokens) > 256:
        # Truncate to the maximum token length
        ground_truth_tokens = ground_truth_tokens[:256]

        # Convert the tokens back to a string
        ground_truth_text = tokenizer.convert_tokens_to_string(ground_truth_tokens) 
    else:
        ground_truth_text = orig_ground_truth_text 
    print('\nground_truth_text:', ground_truth_text)

    generated_text_tokens = tokenizer.tokenize(orig_generated_text)
    if len(generated_text_tokens) > 256:
        # Truncate to the maximum token length
        generated_text_tokens = generated_text_tokens[:256]

        # Convert the tokens back to a string
        generated_text = tokenizer.convert_tokens_to_string(generated_text_tokens) 
    else:
        generated_text = orig_generated_text 
    print('\ngenerated_text:', generated_text)

    # Metric #1 Sentence Similarity using BERT
    print('\nMetric #1')
    cls_dist, mean_dist, max_dist = compute_cosine_distances(
        ground_truth_text, generated_text, model, tokenizer, device
    )
    print('cls_dist=', cls_dist)
    print('mean_dist=', mean_dist)
    print('max_dist=', max_dist)
    
    # Metric #2 BERTScore (sentence-level)
    print('\nMetric #2')
    P_sent, R_sent, F1_sent = get_bertscore(
        ground_truth_text, generated_text
    )
    print('P_sent=', P_sent)
    print('R_sent=', R_sent)
    print('F1_sent=', F1_sent)
    
    # Metric #2 BERTScore (document-level)
    print('\nMetric #3')
    P_doc, R_doc, F1_doc = get_document_bertscore(
        ground_truth_text, generated_text
    )
    print('P_doc=', P_doc)
    print('R_doc=', R_doc)
    print('F1_doc=', F1_doc)


article_id: S1889-836X2016000100006-1

ground_truth_text: a 65 - year - old male patient diagnosed with oi from the study of a daughter also affected . she had no previous family history of the disease . 1 . she reported a history of several fractures ( approximately 6 - 7 ) during childhood and adolescence following minimal trauma , the first at 2 years of age . during adulthood , she had had two new elbow and shoulder fractures . all had been treated conservatively . she was treated with calcium during childhood . he had also had multiple sprains and various muscle ruptures throughout his life . diagnosed as otorrhea , she had been operated on for stapedectomy of both ears . search was performed for osteoporosis detected in bone densitometry ( dxa ) , showing t - scores of - 3 . 4 in the lumbar spine ( l1 - l4 ) , - 3 in the femoral neck and - 2 . 8 in the total . he was asymptomatic . physical examination revealed 162 cm in size , blue sclera and presence of dentinogenesis calla . 